Final Data Preprocessing

Dataset bao gồm nhiều file `.parquet` (dạng phân tán), được xử lý bằng PySpark.

Mục tiêu:
- Làm sạch dữ liệu
- Giảm sparsity
- Loại bỏ nhiễu
- Chuẩn bị cho ML và BERT

Các bước:
1. Load dữ liệu
2. Remove duplicate
3. Xử lý review length
4. Filter user/item
5. Handle imbalance
6. Verified reviews
7. So sánh before vs after

Load dữ liệu parquet

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FinalPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 17:59:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Load toàn bộ folder parquet
df = spark.read.parquet("/kaggle/input/datasets/datdong123/amazon-clean-v2/amazon_reviews_cleaned_v3")

df.printSchema()
df.show(5)

root
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: integer (nullable = true)
 |-- verified_purchase: integer (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- helpful_ratio: double (nullable = true)
 |-- review_length: integer (nullable = true)
 |-- product_category: string (nullable = true)



+----------+-----------+--------------+--------------+--------------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+-------------------+-------------+----------------+
|product_id|customer_id|     review_id|product_parent|       product_title|star_rating|helpful_votes|total_votes|vine|verified_purchase|     review_headline|         review_body|review_date|      helpful_ratio|review_length|product_category|
+----------+-----------+--------------+--------------+--------------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+-------------------+-------------+----------------+
|0061928356|   20268060|R3115J8Q83V3KH|     272250770|Secret Daughter: ...|          5|            1|          1|   0|                1|Heart wrenching s...|As someone famili...| 2013-03-27|                1.0|          416|           Books|
|0066214130|   37388508|R1MWBLKR

In [5]:
# Lưu bản raw để so sánh
df_raw = df
print("Initial count:", df.count())

Initial count: 31862547


2. Remove Duplicate

In [6]:
df_dedup = df.dropDuplicates(["customer_id", "product_id", "review_body"])
total_before = df.count()
total_after = df_dedup.count()

print("Before:", total_before)
print("After:", total_after)
print("Removed:", total_before - total_after)

Before: 31862547
After: 31853015
Removed: 9532


3. Review Length + Remove Outlier

In [7]:
from pyspark.sql.functions import length, col

df.select("review_length").show(5)

# Remove review quá ngắn
df = df.filter(col("review_length") >= 10)

# Remove outlier (P99)
quantiles = df.approxQuantile("review_length", [0.99], 0.01)
upper_bound = quantiles[0]

print("Upper bound:", upper_bound)

df = df.filter(col("review_length") <= upper_bound)

+-------------+
|review_length|
+-------------+
|          416|
|         1192|
|         1080|
|         2293|
|          728|
+-------------+
only showing top 5 rows


Upper bound: 3089.0


Save cleaned data

In [8]:
df.write.mode("overwrite").parquet("/content/cleaned_data")